# Genome scale metabolic model analysis

Model from: Model-driven engineering of Cutaneotrichosporon oleaginosus ATCC 20509 for improved microbial oil production by Zeynep Efsun Duman-Özdamar 

Analysis by: Amber van Doorn for MSc internship WFBR

## Requirements before running the code
If any modules to be imported are not present, please download these using !pip install {modeule_name}. The code block below can be altered and uncommented to install for example the cobra, logging and escher packages.
<br>
The file "iNP636_Coleaginosus_ATCC20509_mod_names_V2.xml, available on GitHub should be in the same directory as this notebook. Otherwise the path of the data_dir can be changed to the correct folder.

In [80]:
#!pip install cobra
#!pip install logging
#!pip install escher

## Loading model and printing characteristics

In [81]:
from cobra.io import read_sbml_model, save_json_model
from cobra import Model, Reaction, Metabolite
from cobra.flux_analysis import flux_variability_analysis, single_gene_deletion, single_reaction_deletion, double_gene_deletion, double_reaction_deletion
from pathlib import Path
import logging
import escher
from escher import Builder
from IPython.display import display
import copy
import pandas as pd

# Set path to your model directory
data_dir = Path(r"")
model_path = data_dir / "iNP636_Coleaginosus_ATCC20509_mod_names_V2.xml"

model = read_sbml_model(str(model_path))

#Print some model statistics
print("Model loaded successfully!")
print("Model ID:", model.id)
print("Reactions:", len(model.reactions))
print("Metabolites:", len(model.metabolites))
print("Genes:", len(model.genes))


Model loaded successfully!
Model ID: Cryptococcus_curvatus_model_xml
Reactions: 1556
Metabolites: 1379
Genes: 637


## Defining useful functions for analyzing the model, metabolites and fluxes

In [82]:
#function for printing medium metabolites

def print_medium_metabolites(model, medium_dict=None):
    """
    Print the metabolites present in a COBRA medium dictionary.
    Parameters:
    - model: cobra.Model
    - medium_dict: dict of {exchange_rxn_id: bound} (default: model.medium)
    """
    if medium_dict is None:
        medium_dict = model.medium
    
    print(f"{'Reaction ID':25s} {'Metabolite ID':20s} {'Metabolite Name':40s} {'Bound':>10s} {'Flux':>10s}")
    print("-" * 105)
    
    for rxn_id, bound in medium_dict.items():
        try:
            rxn = model.reactions.get_by_id(rxn_id)
        except KeyError:
            print(f"{rxn_id:25s} {'N/A':20s} {'Reaction not found in model':40s} {bound:10.2f} {'N/A':>10s}")
            continue
        
        # Get the actual flux if a solution exists
        flux_val = "N/A"
        try:
            flux_val = f"{solution.fluxes[rxn_id]:10.4f}"
        except:
            pass
        
        # Find the external metabolite (usually ends with _e)
        external_mets = [met for met in rxn.metabolites if met.compartment == 'e']
        
        if external_mets:
            # Print only the external metabolite(s)
            for met in external_mets:
                print(f"{rxn_id:25s} {met.id:20s} {met.name:40s} {bound:10.2f} {flux_val:>10s}")
        else:
            # If no external metabolite found, print the first one
            met = list(rxn.metabolites.keys())[0]
            print(f"{rxn_id:25s} {met.id:20s} {met.name:40s} {bound:10.2f} {flux_val:>10s}")
    
    print("--------------------------------------------------------------------------------")


def print_reaction(reaction_id):
    rxn = model.reactions.get_by_id(reaction_id)

    flux = solution.fluxes.get(rxn.id, 0)
    print(f"=== {rxn.id} — {rxn.name} ===")
    print(f"Flux: {flux}")
        # Show metabolites in the reaction
    for met, coeff in rxn.metabolites.items():
        sign = "+" if coeff > 0 else ""
        print(f"   {sign}{coeff} {met.name}")
    print()


def give_involved_reactions(model, solution, metabolite_ids, metabolite_name="metabolite"):
    
    # Get metabolite objects
    metabolites = [model.metabolites.get_by_id(x) for x in metabolite_ids]
    print(f"Objective value: {solution.objective_value}\n")
    
    all_reactions = set()
    # Collect all reactions involving any of the metabolites
    for m in metabolites:
        for rxn in m.reactions:
            all_reactions.add(rxn)
    
    # Print the flux for each reaction
    print(f"=== ALL REACTIONS INVOLVING {metabolite_name.upper()} ===\n")
    for rxn in sorted(all_reactions, key=lambda r: r.id):
        flux = solution.fluxes.get(rxn.id, 0)
        print(f"=== {rxn.id} — {rxn.name} ===")
        print(f"Flux: {flux}")
        # Show metabolites in the reaction
        for met, coeff in rxn.metabolites.items():
            sign = "+" if coeff > 0 else ""
            print(f"   {sign}{coeff} {met.name}")
        print()
        
    return all_reactions





def calculate_fluxes(all_reactions, model, solution, metabolite_ids, metabolite_name="metabolite"):
    threshold_percent = 1.0
    # Calculate total production and consumption
    total_production = 0
    total_consumption = 0
    for rxn in all_reactions:
        flux = solution.fluxes.get(rxn.id, 0)
        for met, coeff in rxn.metabolites.items():
            if met.id in metabolite_ids:
                net_change = coeff * flux
                if net_change > 0:
                    total_production += net_change
                elif net_change < 0:
                    total_consumption += (-net_change)
    
    print(f"Total {metabolite_name} production flux: {total_production}")
    print(f"Total {metabolite_name} consumption flux: {total_consumption}")
    
    # Separate summary for production and consumption
    production_rows = []
    consumption_rows = []
    
    for rxn in all_reactions:
        flux = solution.fluxes.get(rxn.id, 0)
        
        # Calculate net production/consumption for this reaction
        production_in_rxn = 0
        consumption_in_rxn = 0
        
        for met, coeff in rxn.metabolites.items():
            if met.id in metabolite_ids:
                net_change = coeff * flux
                if net_change > 0:
                    production_in_rxn += net_change
                elif net_change < 0:
                    consumption_in_rxn += (-net_change)
        
        # Add to production summary
        if production_in_rxn > 0:
            if total_production > 0:
                percent = (production_in_rxn / total_production) * 100
            else:
                percent = 0
            
            production_rows.append({
                "Reaction ID": rxn.id,
                "Pathway / Reaction Name": rxn.name,
                "Flux (mmol/gDW/hr)": flux,
                f"{metabolite_name.title()} Production (mmol/gDW/hr)": production_in_rxn,
                "% of Total Flux": percent
            })
        
        # Add to consumption summary
        if consumption_in_rxn > 0:
            if total_consumption > 0:
                percent = (consumption_in_rxn / total_consumption) * 100
            else:
                percent = 0
            
            consumption_rows.append({
                "Reaction ID": rxn.id,
                "Pathway / Reaction Name": rxn.name,
                "Flux (mmol/gDW/hr)": flux,
                f"{metabolite_name.title()} Consumption (mmol/gDW/hr)": consumption_in_rxn,
                "% of Total Flux": percent
            })
    
    df_production = pd.DataFrame(production_rows)
    df_consumption = pd.DataFrame(consumption_rows)
    
    # Sort by contribution
    if not df_production.empty:
        df_production = df_production.sort_values("% of Total Flux", ascending=False)
    if not df_consumption.empty:
        df_consumption = df_consumption.sort_values("% of Total Flux", ascending=False)
    
    print("\n" + "="*100)
    print(f"{metabolite_name.upper()} PRODUCTION SUMMARY")
    print("="*100)
    if not df_production.empty:
        print(df_production.to_string(index=False))
    else:
        print("No production reactions found.")
    
    print("\n" + "="*100)
    print(f"{metabolite_name.upper()} CONSUMPTION SUMMARY")
    print("="*100)
    if not df_consumption.empty:
        print(df_consumption.to_string(index=False))
    else:
        print("No consumption reactions found.")
    
    # Print detailed reactions for production pathways above threshold
    print("\n" + "="*100)
    print(f"DETAILED PRODUCTION REACTIONS (>{threshold_percent}% contribution)")
    print("="*100)
    
    if not df_production.empty:
        for _, row in df_production.iterrows():
            if row["% of Total Flux"] > threshold_percent:
                rxn_id = row["Reaction ID"]
                rxn = model.reactions.get_by_id(rxn_id)
                
                print(f"\n{rxn_id} — {rxn.name}")
                print(f"Flux: {row['Flux (mmol/gDW/hr)']:.4f} mmol/gDW/hr")
                print(f"{metabolite_name.title()} Production: {row[f'{metabolite_name.title()} Production (mmol/gDW/hr)']:.4f} mmol/gDW/hr")
                print(f"Contribution: {row['% of Total Flux']:.2f}%")
                print(f"\nReaction equation:")
                print(f"  {rxn.reaction}")
                print(f"\nDetailed stoichiometry:")
                
                # Separate reactants and products
                reactants = []
                products = []
                for met, coeff in rxn.metabolites.items():
                    met_str = f"{abs(coeff):.2f} {met.name} [{met.id}]"
                    if coeff < 0:
                        reactants.append(met_str)
                    else:
                        products.append(met_str)
                
                print("  Reactants:")
                for r in reactants:
                    print(f"    - {r}")
                print("  Products:")
                for p in products:
                    print(f"    + {p}")
                print("-" * 100)
    else:
        print("No production reactions above threshold.")
    
    # Print detailed reactions for consumption pathways above threshold
    print("\n" + "="*100)
    print(f"DETAILED CONSUMPTION REACTIONS (>{threshold_percent}% contribution)")
    print("="*100)
    
    if not df_consumption.empty:
        for _, row in df_consumption.iterrows():
            if row["% of Total Flux"] > threshold_percent:
                rxn_id = row["Reaction ID"]
                rxn = model.reactions.get_by_id(rxn_id)
                
                print(f"\n{rxn_id} — {rxn.name}")
                print(f"Flux: {row['Flux (mmol/gDW/hr)']:.4f} mmol/gDW/hr")
                print(f"{metabolite_name.title()} Consumption: {row[f'{metabolite_name.title()} Consumption (mmol/gDW/hr)']:.4f} mmol/gDW/hr")
                print(f"Contribution: {row['% of Total Flux']:.2f}%")
                print(f"\nReaction equation:")
                print(f"  {rxn.reaction}")
                print(f"\nDetailed stoichiometry:")
                
                # Separate reactants and products
                reactants = []
                products = []
                for met, coeff in rxn.metabolites.items():
                    met_str = f"{abs(coeff):.2f} {met.name} [{met.id}]"
                    if coeff < 0:
                        reactants.append(met_str)
                    else:
                        products.append(met_str)
                
                print("  Reactants:")
                for r in reactants:
                    print(f"    - {r}")
                print("  Products:")
                for p in products:
                    print(f"    + {p}")
                print("-" * 100)
    else:
        print("No consumption reactions above threshold.")
    
    return df_production, df_consumption



In [83]:
biomass_reaction_abundant = "Biomass_nitrogen_abundant"
biomass_reaction_depletion = "Biomass_nitrogen_depletion"
lipid_reaction = "Ex_lipid_body_cytosol"

print("Simulations running on medium containing:") 
print_medium_metabolites(model)


model.objective = biomass_reaction_abundant
solution = model.optimize()
print(f"The solution is : {solution.objective_value} gDW · h⁻¹ for {model.objective.expression}")

model.objective = lipid_reaction
solution = model.optimize()
print(f"The solution is : {solution.objective_value} mmol · gDW⁻¹ · h⁻¹ for {model.objective.expression}")
print("\n")

#all other solutions will be based on this objective as it is the last one defined:
model.objective = biomass_reaction_depletion
solution = model.optimize()
print(f"The solution is : {solution.objective_value} gDW · h⁻¹ for {model.objective.expression}")




Simulations running on medium containing:
Reaction ID               Metabolite ID        Metabolite Name                               Bound       Flux
---------------------------------------------------------------------------------------------------------
r_51_exchange             glc_D_e              D-glucose [extracellular]                     20.00   -20.0000
r_77_exchange             h_e                  H+ [extracellular]                          1000.00    -0.0663
r_82_exchange             C14818_e             iron(2+) [extracellular]                    1000.00     0.0000
r_128_exchange            25805_e              oxygen [extracellular]                      1000.00    -0.9298
r_134_exchange            pi_e                 phosphate [extracellular]                   1000.00    -0.0127
r_136_exchange            k_e                  potassium [extracellular]                   1000.00     0.0000
r_144_exchange            na1_e                sodium [extracellular]             

# The path to serine in C. oleaginosus

In [84]:
serine_ids = ["ser_L_c", "ser_L_r", "ser_L_m", "ser_L_e"]
all_serine_reactions = give_involved_reactions(model, solution, serine_ids, metabolite_name="serine")


P_glycerate_ids = ["132960_c"]
all_3P_glycerate_reactions = give_involved_reactions(model, solution, P_glycerate_ids, metabolite_name="3-phosphoglycerate")


print("\n There are only two reactions involving 3-phosphoglycerate, neither one is involved in the SerA/Ser33 reaction")
print("L-serine is mostly produced via r_0540, and consumed via r_0664")

Objective value: 0.03789635860143442

=== ALL REACTIONS INVOLVING SERINE ===

=== r_0338 — cystathionine beta-synthase ===
Flux: 0.0007161333227778032
   -1.0 L-homocysteine [cytoplasm]
   -1.0 L-serine [cytoplasm]
   +1.0 L-cystathionine [cytoplasm]
   +1.0 H2O [cytoplasm]

=== r_0539 — glycine hydroxymethyltransferase ===
Flux: 0.008068210754583893
   -1.0 THF [cytoplasm]
   -1.0 L-serine [cytoplasm]
   +1.0 5,10-methylenetetrahydrofolate(2-) [cytoplasm]
   +1.0 L-glycine [cytoplasm]
   +1.0 H2O [cytoplasm]

=== r_0540 — glycine hydroxymethyltransferase ===
Flux: -2.539121536142311
   -1.0 THF [mitochondrion]
   -1.0 L-serine [mitochondrion]
   +1.0 5,10-methylenetetrahydrofolate(2-) [mitochondrion]
   +1.0 L-glycine [mitochondrion]
   +1.0 H2O [mitochondrion]

=== r_0664 — L-serine deaminase ===
Flux: 2.510283333056377
   -1.0 L-serine [cytoplasm]
   +1.0 ammonium [cytoplasm]
   +1.0 pyruvate [cytoplasm]

=== r_0665 — L-serine dehydrogenase ===
Flux: 0.0
   -1.0 NADP(+) [cytoplasm]


## Adding missing reactions
3P-D-Glycerate (M_132960_c) 
             to             #add reaction!
3-phospho-hydroxypyruvate [cytoplasm] = "M_C03232_c" 
             to         (r_0893
"M_C01005_c" name="3-phospho-serine [cytoplasm]" 
                to             #add reaction!
serine = M_ser_L_c"

In [85]:

reaction1 = Reaction('r_SerA')
reaction1.name = 'D-3-phosphoglycerate dehydrogenase'
reaction1.lower_bound = -1000
reaction1.upper_bound = 1000

reaction1.add_metabolites({
    model.metabolites.get_by_id("132960_c"): -1.0,
    model.metabolites.get_by_id("nad_c"): -1.0,
    model.metabolites.get_by_id("C03232_c"):  1.0,
    model.metabolites.get_by_id("nadh_c"): 1.0,
    model.metabolites.get_by_id("h_c"): 1.0
})

model.add_reactions([reaction1])

print("Added:", reaction1.id, reaction1.reaction)

reaction2 = Reaction('r_SerC')
reaction2.name = 'phosphoserine_phosphatase'
reaction2.lower_bound = 0
reaction2.upper_bound = 1000

reaction2.add_metabolites({
    model.metabolites.get_by_id("C01005_c"): -1.0,
    model.metabolites.get_by_id("h2o_c"): -1.0,
    model.metabolites.get_by_id("ser_L_c"): 1.0,
    model.metabolites.get_by_id("pi_c"): 1.0
})

model.add_reactions([reaction2])

    
print("Added:", reaction2.id, reaction2.reaction)


# Allow serine export as well as import:
model.reactions.r_1393.lower_bound = -1000
model.reactions.r_1393.upper_bound = 1000


Added: r_SerA 132960_c + nad_c <=> C03232_c + h_c + nadh_c
Added: r_SerC C01005_c + h2o_c --> pi_c + ser_L_c


In [86]:
model.objective = "Biomass_nitrogen_depletion" # Get the reaction from the model
solution = model.optimize()

P_glycerate_ids = ["132960_c"]
all_3P_glycerate_reactions = give_involved_reactions(model, solution, P_glycerate_ids, metabolite_name="3-phosphoglycerate")


P_hydorxyglycerate_ids = ["C03232_c"]
all_3P_hydroxylglycerate_reactions = give_involved_reactions(model, solution, P_hydorxyglycerate_ids, metabolite_name="3-phospho hydroxyglycerate")

phosphoserine_id = ["C01005_c"]
all_phosphoserine_reactions = give_involved_reactions(model, solution, phosphoserine_id, metabolite_name="3-Phosphoserine")


print("Now using serine transport as the model objective:")


model.objective ="r_105_exchange" #serine exchange reaction
solution = model.optimize()

print("Max serine export:", solution.objective_value)

P_glycerate_ids = ["132960_c"]
all_3P_glycerate_reactions = give_involved_reactions(model, solution, P_glycerate_ids, metabolite_name="3-phosphoglycerate")


P_hydorxyglycerate_ids = ["C03232_c"]
all_3P_hydroxylglycerate_reactions = give_involved_reactions(model, solution, P_hydorxyglycerate_ids, metabolite_name="3-phospho hydroxyglycerate")


phosphoserine_id = ["C01005_c"]
all_phosphoserine_reactions = give_involved_reactions(model, solution, phosphoserine_id, metabolite_name="3-Phosphoserine")


Objective value: 0.037896358601432

=== ALL REACTIONS INVOLVING 3-PHOSPHOGLYCERATE ===

=== r_0865 — phosphoglycerate kinase ===
Flux: 39.85974803844282
   -1.0 1,3-bisphospho-D-glycerate [cytoplasm]
   -1.0 ADP [cytoplasm]
   +1.0 3-phosphoglycerate [cytoplasm]
   +1.0 ATP [cytoplasm]

=== r_0866 — phosphoglycerate mutase ===
Flux: 39.85974803844282
   -1.0 3-phosphoglycerate [cytoplasm]
   +1.0 2-phospho-D-glyceric acid [cytoplasm]

=== r_SerA — D-3-phosphoglycerate dehydrogenase ===
Flux: 0.0
   -1.0 3-phosphoglycerate [cytoplasm]
   -1.0 NAD [cytoplasm]
   +1.0 3-phospho-hydroxypyruvate [cytoplasm]
   +1.0 NADH [cytoplasm]
   +1.0 H+ [cytoplasm]

Objective value: 0.037896358601432

=== ALL REACTIONS INVOLVING 3-PHOSPHO HYDROXYGLYCERATE ===

=== r_0893 — phosphoserine transaminase ===
Flux: 0.0
   -1.0 3-phospho-hydroxypyruvate [cytoplasm]
   -1.0 L-glutamate [cytoplasm]
   +1.0 3-phospho-serine [cytoplasm]
   +1.0 2-oxoglutarate [cytoplasm]

=== r_SerA — D-3-phosphoglycerate dehydr

## FVA: Flux variability analysis

To show the solution to the biomass_depletion reaction can also have an optimal solution with different fluxes, FVA is used. 

In [87]:
#First 10 reactions for FVA

model.objective = "Biomass_nitrogen_depletion" # Get the reaction from the model
solution = model.optimize()

reactions_of_interest = ["r_105_exchange", "r_SerA", "r_SerC", "r_0893"]

fva_result = flux_variability_analysis(
    model,
    reaction_list=reactions_of_interest,
    fraction_of_optimum=0.95  # 95% of max biomass
)

print(fva_result)

print("fluxes for the r_105_exchange reaction at max")


model.objective = "r_105_exchange" # Get the reaction from the model
solution = model.optimize()

reactions_of_interest = ["r_105_exchange", "r_SerA", "r_SerC", "r_0893"]

fva_result = flux_variability_analysis(
    model,
    reaction_list=reactions_of_interest,
    fraction_of_optimum=0.95  # 95% of max biomass
)

print(fva_result)



                minimum   maximum
r_105_exchange      0.0  1.589190
r_SerA              0.0  1.088889
r_SerC              0.0  1.088889
r_0893              0.0  1.088889
fluxes for the r_105_exchange reaction at max
                     minimum    maximum
r_105_exchange  3.800000e+00   4.000000
r_SerA          1.260490e-15  19.174074
r_SerC          0.000000e+00  19.174074
r_0893          0.000000e+00  19.174074


## SerA / Ser33 deletions 

As SerA and SerC are not required for growth, knockouts of a different gene need to be made in order to create a true serine auxotrophic strain. 

In [102]:

model.objective = "Biomass_nitrogen_depletion" # Get the reaction from the model
solution = model.optimize()

with model:
    model.reactions.r_0893.knock_out()
    print('3-phospho-hydroxypyruvate to 3-phospho-serine knocked out ', model.optimize())


with model:
    rxn1 = model.reactions.get_by_id("r_0539")
    rxn2 = model.reactions.get_by_id("R_new1")

    rxn1.knock_out()
    rxn2.knock_out()

    sol = model.optimize()
    print("Growth after gly -> ser + SerA KO:", sol.objective_value)
    
#add serine to medium:


<Solution 0.038 at 0x22692a77540>


In [93]:
glycine_ids = ["gly_c", "gly_m", "gly_e"]
glycine_mets = [model.metabolites.get_by_id(x) for x in glycine_ids]

all_glycine_reactions = set()

# Collect all reactions involving any of the glycine metabolites
for m in glycine_mets:
    for rxn in m.reactions:
        all_glycine_reactions.add(rxn)

        
# Print the flux for each reaction
for rxn in sorted(all_glycine_reactions, key=lambda r: r.id):
    flux = solution.fluxes.get(rxn.id, 0)

    print(f"=== {rxn.id} — {rxn.name} ===")
    print(f"Flux: {flux}")

    # Show metabolites in the reaction
    for met, coeff in rxn.metabolites.items():
        sign = "+" if coeff > 0 else ""
        print(f"   {sign}{coeff} {met.name}")

    print()


=== Protein_synthesis — Protein_synthesis ===
Flux: 0.060141520255486774
   -0.0664852221584454 Lys-tRNA(Lys) [cytoplasm]
   -0.0694620895109579 Leu-tRNA(Leu) [cytoplasm]
   -0.0138921402093415 Met-tRNA(Met) [cytoplasm]
   -0.0992307630360826 Ala-tRNA(Ala) [cytoplasm]
   -0.0377079121079914 Arg-tRNA(Arg) [cytoplasm]
   -0.0119074694100499 Cys-tRNA(Cys) [cytoplasm]
   -0.0317541774029665 Phe-tRNA(Phe) [cytoplasm]
   -0.0357224082301496 Pro-tRNA(Pro) [cytoplasm]
   -0.0208383491604373 His-tRNA(His) [cytoplasm]
   -2.77692850047805e-07 Gly-tRNA(Gly) [cytoplasm]
   -0.0744216838128117 Glu-tRNA(Glu) [cytoplasm]
   -0.0476298776401995 Asp-tRNA(Asp) [cytoplasm]
   -0.00777539980133854 Trp-tRNA(Trp) [cytoplasm]
   -0.0476298776401995 Ile-tRNA(Ile) [cytoplasm]
   -0.0267920838654623 Tyr-tRNA(Tyr) [cytoplasm]
   -0.0704534529856286 Ser-tRNA(Ser) [cytoplasm]
   -0.0545749758198951 Thr-tRNA(Thr) [cytoplasm]
   -0.0714448164602992 Val-tRNA(Val) [cytoplasm]
   -0.0744216838128117 Gln-tRNA(Gln) [cyto

This says us that from glycine all serine is made, not from the glycolysis to 3P-D-Glycerate route. This route is now knocked out.

In [101]:
solution = model.optimize()


with model:
    model.reactions.r_0540.knock_out()
    model.reactions.r_0539.knock_out()
    print('3-phospho-hydroxypyruvate to 3-phospho-serine knocked out ', model.optimize())
# --> this alone still allows for growth


with model:
    model.reactions.r_SerA.knock_out()
    print('3-phospho-hydroxypyruvate to 3-phospho-serine knocked out ', model.optimize())

#--> this alone also does not grant serine auxotrophy
# 0.37/0.38 = 0.973 so 97.3% of the biomass can still be obtained with 


#combining all knockouts:
with model:
    model.reactions.r_0540.knock_out()
    model.reactions.r_0539.knock_out()
    model.reactions.r_SerA.knock_out()
    print('3-phospho-hydroxypyruvate to 3-phospho-serine knocked out ', model.optimize())






3-phospho-hydroxypyruvate to 3-phospho-serine knocked out  <Solution 0.037 at 0x22692a74600>
3-phospho-hydroxypyruvate to 3-phospho-serine knocked out  <Solution 0.038 at 0x22692a75400>
3-phospho-hydroxypyruvate to 3-phospho-serine knocked out  <Solution -0.000 at 0x22692a74600>
